In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import numpy as np
import pandas as pd

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
project_path = "/content/drive/MyDrive/TrustPilot_Project"
preprocessed_path = project_path + "/data/preprocessed"
embeddings_path = project_path + "/embeddings"

In [ ]:
# Load preprocessed dataset
with open(os.path.join(preprocessed_path, "dataset_preprocessed.pkl"), "rb") as f:
    df = pickle.load(f)


In [ ]:
# load TF-IDF Embedding
#with open(os.path.join(embeddings_path, "tfidf_vectorizer.pkl"), "rb") as f:
  #  vectorizer = pickle.load(f)

#with open(os.path.join(embeddings_path, "emb_tfidf.pkl"), "rb") as f:
  #  tfidf_embeddings = pickle.load(f)


In [ ]:
df["tokens"].head(5)

,tokens
0,"[bonjour, faire, an, membre, showroopriv, jama..."
1,"[vente, lacost, article, manquant, photo, pren..."
2,"[vente, lacost, honteux, article, erroné, arti..."
3,"[commander, mule, marque, moosefield, déçu, pr..."
4,"[commande, téléphon, etat, livraison, vieux, t..."


In [ ]:
# Create a cleaned text column from tokens
df["cleaned_text"] = df["tokens"].apply(lambda x: " ".join(x))

In [ ]:
df["cleaned_text"].head()

,cleaned_text
0,bonjour faire an membre showroopriv jamais sou...
1,vente lacost article manquant photo prendre ar...
2,vente lacost honteux article erroné article ma...
3,commander mule marque moosefield déçu produit ...
4,commande téléphon etat livraison vieux télépho...


In [ ]:
# Combine French stopwords + product names
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

french_stopwords = stopwords.words('french')
product_stopwords = ['montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille',
                     'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier',
                     'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur',
                     'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes',
                     'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin',
                     'bague', 'lampe', 'lampes', 'abat-jour', 'lampadaire', 'parfum',
                     'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques',
                     'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo',
                     'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire',
                     'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones',
                     'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts',
                     'hortensias', 'orchidée', 'sacs', 'plant', 'reconditionné']

all_stopwords = french_stopwords + product_stopwords

vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
    ngram_range=(1,2),
    stop_words=all_stopwords
)
tfidf_emb = vectorizer.fit_transform(df['cleaned_text'])



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['jour'] not in stop_words.
  warnings.warn(


In [ ]:
# LDA model
lda_model = LatentDirichletAllocation(
    n_components=10,
    learning_method='online',
    random_state=42
)

lda_model.fit(tfidf_emb)

LatentDirichletAllocation(learning_method='online', random_state=42)

In [ ]:
# Afficher les top mots de chaque topic
def print_top_words(model, feature_names, n_top_words=10):
    topic_words = {}
    for topic_idx, topic in enumerate(model.components_):
        top_features = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_features)}")
        topic_words[topic_idx] = top_features
    return topic_words

feature_names = vectorizer.get_feature_names_out()
topic_words = print_top_words(lda_model, feature_names)

Topic 1: decue, conforme attendre, recommander marque, bien recommander, miroir, tacher, fermoir, enchanté, recu commande, joli bijou
Topic 2: conforme attente, conforme, attente, commande conforme, satisfaisant, qualité moyen, assiette, être parfait, parfait conforme, bien protéger
Topic 3: satisfaite, top, beau produit, livraison conforme, décevoir produit, satisfaite achat, bon livraison, commande parfait, taille parfait, produit super
Topic 4: conformer, rapide article, article conformer, rien parfait, avance, achat livraison, jamais décevoir, aimer beaucoup, conformer description, correspondre parfaitement
Topic 5: livraison temps, satisfaite commande, rapide prévoir, qualité article, bien ensemble, envoi rapide, raisonnable, description livraison, satisfaire commande, respecter produit
Topic 6: parfait, bel, livraison respecter, nickel, bel qualité, respecter, content produit, parfait rien, bel article, arriver rapidement
Topic 7: commande, colis, client, service, recevoir, livra

In [ ]:
# Taille de chaque topic
#Dominant topic par document
topics = np.argmax(lda_model.transform(tfidf_emb), axis=1)
# Compter les documents par topic
topics_size = pd.Series(topics).value_counts().sort_index()
topics_size

,count
0,28
1,76
2,60
3,66
4,62
5,100
6,7726
7,26
8,143
9,6790


In [ ]:
# Visualisation des topics
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.bar(topics_size.index + 1, topics_size.values, color='skyblue')
plt.xlabel('Topic')
plt.ylabel('Number of documents')
plt.title('LDA Topic Sizes')
plt.xticks(topics_size.index + 1)
plt.show()

In [ ]:
#Diversité des topics
def topic_diversity(topic_words):
    words = []
    for w in topic_words.values():
        words.extend(w)
    return len(set(words)) / len(words)

diversity_score = topic_diversity(topic_words)
print(f"Diversité des topics: {diversity_score:.2f}")


Diversité des topics: 0.98


In [ ]:
!pip install gensim

from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

top_words_list = list(topic_words.values())

clean_topics = []
for topic in top_words_list:
    cleaned = []
    for word in topic:
        cleaned.extend(word.split())  # split bigrams
    clean_topics.append(cleaned)

tokenized_texts = df["tokens"].tolist()
dictionary = Dictionary(tokenized_texts)

coherence_model = CoherenceModel(
    topics=clean_topics,
    texts=tokenized_texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("Coherence (c_v):", coherence_score)


Coherence (c_v): 0.3528153049878241


In [ ]:
#topics_df = pd.DataFrame(topic_words).T
#topics_df.to_csv("topics.csv")
